# Notebook 04: End-to-End Fault Detection Demo

**AI Power Electronics Diagnostics — IEEE IES Industrial AI Lab**

This notebook demonstrates the complete fault detection pipeline:

1. **Train** a 1D CNN fault classifier
2. **Deploy** via `SwitchFaultDetector` wrapper
3. **Detect** faults on streaming inverter signals
4. **Harmonic fault** detection with rule-based analysis
5. **Thermal fault** detection with autoencoder
6. **Full diagnostic dashboard** visualization

This is the end-to-end demo that ties the whole framework together.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from datasets.synthetic import InverterFaultSimulator, MotorDriveSimulator, FaultInjector
from models import CNN1DWaveformClassifier, AutoencoderAnomalyDetector
from fault_detection import SwitchFaultDetector, HarmonicFaultDetector, ThermalFaultDetector
from visualization import WaveformPlotter, FaultDashboard

plt.rcParams['figure.dpi'] = 100
device = torch.device('cpu')
print('Framework ready.')

## Part 1: Train and Deploy 1D CNN Switch Fault Detector

In [ ]:
# Quick training for demo
sim = InverterFaultSimulator()
X, y = sim.generate_dataset(n_per_class=100, window_size=1024)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = CNN1DWaveformClassifier(n_channels=6, n_classes=9, window_size=1024)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

from torch.utils.data import DataLoader, TensorDataset
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train.astype('float32')),
                  torch.from_numpy(y_train.astype('int64'))),
    batch_size=64, shuffle=True
)

print('Training CNN (15 epochs demo)...')
for epoch in range(15):
    model.train()
    for X_b, y_b in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1} done')

print('Training complete!')

In [ ]:
# Deploy via SwitchFaultDetector
detector = SwitchFaultDetector(
    model=model,
    f_sample=100_000.0,
    window_size=1024,
    confidence_threshold=0.6
)

# Test on a few samples
test_signals = [('healthy', sim.generate('healthy')[0]),
                ('open_circuit_T1', sim.generate('open_circuit_T1')[0]),
                ('short_circuit', sim.generate('short_circuit')[0])]

print('\n=== Single-Sample Detection ===')
for true_label, sig in test_signals:
    window = sig[:, :1024]
    result = detector.detect(window)
    status = '✓' if result['fault_type'] == true_label else '✗'
    print(f'  {status} True: {true_label:<20} Predicted: {result["fault_type"]:<20} Conf: {result["confidence"]*100:.1f}%')

## Part 2: Streaming Detection

In [ ]:
# Simulate a continuous signal: healthy for first half, then OC fault injected
s_long_h, _ = sim.generate('healthy')      # (6, ~200000)
s_long_f, _ = sim.generate('open_circuit_T3')

half = s_long_h.shape[1] // 2
stream = np.concatenate([s_long_h[:, :half], s_long_f[:, :half]], axis=1)

# Streaming detection (hop = 512 = 50% overlap)
detections = detector.streaming_detect(stream, hop_size=512)

# Extract predicted labels and timestamps
times_ms = [d['time_start_s'] * 1000 for d in detections]
pred_labels = [d['label'] for d in detections]

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(times_ms, pred_labels, '.', markersize=4, color='#1f77b4')
ax.axvline(half / 100_000 * 1000, color='#d62728', linestyle='--', label='Fault injection')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Predicted Class')
ax.set_title('Streaming Fault Detection')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 3: Harmonic Fault Detector

In [ ]:
motor_sim = MotorDriveSimulator()
harm_detector = HarmonicFaultDetector(f_sample=50_000, f_fund=50.0)

test_cases = {
    'healthy':  motor_sim.generate('healthy')[0][0],
    'itsc':     motor_sim.generate('itsc')[0][0],
    'bearing':  motor_sim.generate('bearing')[0][0],
}

print('Harmonic Fault Analysis:')
for name, sig in test_cases.items():
    result = harm_detector.analyze(sig)
    print(f'  {name:<12} | THD-F: {result.thd_f:.2f}% | Fault: {result.fault_type} | Severity: {result.severity} | ITSC: {result.itsc_detected}')

## Part 4: Thermal Fault Detector (Autoencoder)

In [ ]:
ae_model = AutoencoderAnomalyDetector(n_channels=3, window_size=1024)
ae_optimizer = torch.optim.Adam(ae_model.parameters(), lr=1e-3)

# Train AE on healthy motor signals only
X_motor, y_motor = motor_sim.generate_dataset(n_per_class=80, window_size=1024)
X_healthy_m = X_motor[y_motor == 0].astype('float32')

print('Training autoencoder on healthy motor signals...')
X_h_t = torch.from_numpy(X_healthy_m)
for epoch in range(20):
    ae_model.train()
    perm = torch.randperm(len(X_h_t))
    for i in range(0, len(X_h_t), 32):
        batch = X_h_t[perm[i:i+32]]
        ae_optimizer.zero_grad()
        loss = ae_model.reconstruction_loss(batch)
        loss.backward()
        ae_optimizer.step()

# Calibrate threshold
threshold = ae_model.set_threshold(X_h_t, percentile=95)
print(f'Autoencoder threshold: {threshold:.6f}')

# Test
thermal_detector = ThermalFaultDetector(f_sample=50_000, autoencoder=ae_model, ae_threshold=threshold)
s_h, _ = motor_sim.generate('healthy')
s_ot, _ = motor_sim.generate('overtemperature')

thermal_detector.set_baseline(s_h[0])

res_h  = thermal_detector.detect(s_h[:, :1024])
res_ot = thermal_detector.detect(s_ot[:, :1024])
print(f'  Healthy      | is_fault: {res_h.is_fault}  | severity: {res_h.severity} | AE score: {res_h.autoencoder_score:.6f}')
print(f'  Overtemp     | is_fault: {res_ot.is_fault} | severity: {res_ot.severity} | AE score: {res_ot.autoencoder_score:.6f}')

## Part 5: Full Diagnostic Dashboard

In [ ]:
from datasets.synthetic.inverter_fault_sim import INVERTER_FAULT_LABELS
class_names = list(INVERTER_FAULT_LABELS.keys())

test_sig, _ = sim.generate('open_circuit_T3')
test_window = test_sig[:, :1024]

result = detector.detect(test_window)
probs = np.array(list(result['probabilities'].values()))

dashboard = FaultDashboard(
    f_sample=100_000.0,
    f_fund=50.0,
    class_names=class_names
)

fig = dashboard.plot(
    signal=test_window,
    probabilities=probs,
    predicted_fault=result['fault_type'],
    channel_idx=3,
    title='Inverter Diagnostic Dashboard — Open Circuit T3 Fault'
)
plt.show()
print('Notebook 04 complete — Full pipeline demonstrated!')